# CUDA Programming: Vector Addition and Matrix Multiplication
This notebook demonstrates CUDA implementations for large vector addition and matrix multiplication, comparing their performance with sequential CPU implementations.

In [ ]:
!nvcc --version

## 1. Addition of Two Large Vectors

In [ ]:
%%writefile vector_add.cu
#include <iostream>
#include <cuda_runtime.h>
#include <chrono>

using namespace std;

__global__ void vectorAdd(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}

void sequentialAdd(int *a, int *b, int *c, int n) {
    for (int i = 0; i < n; i++) {
        c[i] = a[i] + b[i];
    }
}

int main() {
    int n = 1 << 25; // ~33 million elements
    size_t size = n * sizeof(int);

    int *h_a, *h_b, *h_c_cpu, *h_c_gpu;
    h_a = (int *)malloc(size);
    h_b = (int *)malloc(size);
    h_c_cpu = (int *)malloc(size);
    h_c_gpu = (int *)malloc(size);

    for (int i = 0; i < n; i++) {
        h_a[i] = i % 100;
        h_b[i] = (i * 2) % 100;
    }

    // Sequential
    auto start = chrono::high_resolution_clock::now();
    sequentialAdd(h_a, h_b, h_c_cpu, n);
    auto end = chrono::high_resolution_clock::now();
    chrono::duration<double> seq_duration = end - start;
    cout << "Sequential Vector Addition Time: " << seq_duration.count() << " s" << endl;

    // CUDA
    int *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, size);
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (n + threadsPerBlock - 1) / threadsPerBlock;

    start = chrono::high_resolution_clock::now();
    vectorAdd<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_c, n);
    cudaDeviceSynchronize();
    end = chrono::high_resolution_clock::now();
    chrono::duration<double> gpu_duration = end - start;
    cout << "CUDA Vector Addition Time (Kernel only): " << gpu_duration.count() << " s" << endl;

    cudaMemcpy(h_c_gpu, d_c, size, cudaMemcpyDeviceToHost);

    // Verify
    bool success = true;
    for (int i = 0; i < n; i++) {
        if (h_c_cpu[i] != h_c_gpu[i]) {
            success = false;
            break;
        }
    }
    cout << "Verification: " << (success ? "SUCCESS" : "FAILED") << endl;
    cout << "Speedup: " << seq_duration.count() / gpu_duration.count() << "x" << endl;

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c_cpu);
    free(h_c_gpu);

    return 0;
}

In [ ]:
!nvcc vector_add.cu -o vector_add && ./vector_add

## 2. Matrix Multiplication using CUDA C

In [ ]:
%%writefile matrix_mul.cu
#include <iostream>
#include <cuda_runtime.h>
#include <chrono>

using namespace std;

__global__ void matrixMul(int *a, int *b, int *c, int n) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < n && col < n) {
        int sum = 0;
        for (int k = 0; k < n; k++) {
            sum += a[row * n + k] * b[k * n + col];
        }
        c[row * n + col] = sum;
    }
}

void sequentialMatrixMul(int *a, int *b, int *c, int n) {
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            int sum = 0;
            for (int k = 0; k < n; k++) {
                sum += a[i * n + k] * b[k * n + j];
            }
            c[i * n + j] = sum;
        }
    }
}

int main() {
    int n = 1024; // Small size for CPU comparison, GPU can handle much more
    size_t size = n * n * sizeof(int);

    int *h_a, *h_b, *h_c_cpu, *h_c_gpu;
    h_a = (int *)malloc(size);
    h_b = (int *)malloc(size);
    h_c_cpu = (int *)malloc(size);
    h_c_gpu = (int *)malloc(size);

    for (int i = 0; i < n * n; i++) {
        h_a[i] = rand() % 10;
        h_b[i] = rand() % 10;
    }

    // Sequential
    auto start = chrono::high_resolution_clock::now();
    sequentialMatrixMul(h_a, h_b, h_c_cpu, n);
    auto end = chrono::high_resolution_clock::now();
    chrono::duration<double> seq_duration = end - start;
    cout << "Sequential Matrix Multiplication Time: " << seq_duration.count() << " s" << endl;

    // CUDA
    int *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, size);
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(16, 16);
    dim3 blocksPerGrid((n + 15) / 16, (n + 15) / 16);

    start = chrono::high_resolution_clock::now();
    matrixMul<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_c, n);
    cudaDeviceSynchronize();
    end = chrono::high_resolution_clock::now();
    chrono::duration<double> gpu_duration = end - start;
    cout << "CUDA Matrix Multiplication Time (Kernel only): " << gpu_duration.count() << " s" << endl;

    cudaMemcpy(h_c_gpu, d_c, size, cudaMemcpyDeviceToHost);

    // Verify
    bool success = true;
    for (int i = 0; i < n * n; i++) {
        if (h_c_cpu[i] != h_c_gpu[i]) {
            success = false;
            break;
        }
    }
    cout << "Verification: " << (success ? "SUCCESS" : "FAILED") << endl;
    cout << "Speedup: " << seq_duration.count() / gpu_duration.count() << "x" << endl;

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    free(h_a);
    free(h_b);
    free(h_c_cpu);
    free(h_c_gpu);

    return 0;
}

In [ ]:
!nvcc matrix_mul.cu -o matrix_mul && ./matrix_mul